# Macro Regime Engine

**Project A — Macro PM Resume Track**

Classify U.S. macro regimes with a Hidden Markov Model (HMM), study cross-asset behavior in each regime, and backtest a simple regime-conditional allocator.

| Step | What |
|------|------|
| 1 | Pull macro data (FRED) + cross-asset ETFs (Yahoo) |
| 2 | Engineer monthly features (yield curve, inflation, labor, vol) |
| 3 | Fit 3-state Gaussian HMM |
| 4 | Label regimes: Expansion / Slowdown / Stress |
| 5 | Analyze returns by regime + out-of-sample backtest |

---
**Runs without setup** — click Open in Colab → Run all. No API key required (uses public FRED CSV). Optional: add `FRED_API_KEY` in Colab Secrets for faster data pulls.

In [ ]:
!pip install -q fredapi yfinance hmmlearn plotly

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import yfinance as yf
from fredapi import Fred
from hmmlearn.hmm import GaussianHMM
from sklearn.preprocessing import StandardScaler

plt.style.use("seaborn-v0_8-whitegrid")
sns.set_palette("colorblind")
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

In [ ]:
# --- Configuration ---
START_DATE = "2002-01-01"
END_DATE = None  # None = today
TRAIN_END = "2015-12-31"  # HMM trained through this date; backtest is OOS after

FRED_SERIES = {
    "dgs10": "DGS10",       # 10Y Treasury yield
    "dgs2": "DGS2",         # 2Y Treasury yield
    "cpi": "CPIAUCSL",      # CPI (level → YoY)
    "unrate": "UNRATE",     # Unemployment rate
    "fedfunds": "FEDFUNDS", # Effective Fed Funds
    "vix": "VIXCLS",        # VIX
}

ASSETS = {
    "SPY": "US Equities",
    "TLT": "Long Treasuries",
    "GLD": "Gold",
    "UUP": "US Dollar",
}

REGIME_NAMES = ["Expansion", "Slowdown", "Stress"]
RANDOM_STATE = 42

In [ ]:
def get_fred_api_key():
    """Load FRED API key from Colab Secrets or environment variable."""
    secret_names = ["FRED_API_KEY", "FRED"]  # FRED_API_KEY is preferred

    try:
        from google.colab import userdata
        for name in secret_names:
            try:
                key = userdata.get(name)
                if key:
                    print(f"Using Colab secret: {name}")
                    return key
            except Exception:
                continue
    except ImportError:
        pass

    for name in secret_names:
        key = os.environ.get(name)
        if key:
            print(f"Using environment variable: {name}")
            return key

    return None


api_key = get_fred_api_key()
fred = Fred(api_key=api_key) if api_key else None

if fred:
    print("FRED API connected.")
else:
    print("No API key found — using public FRED CSV downloads (no signup needed).")

## 1. Load data

In [ ]:
def fetch_fred_public_csv(series_map, start, end=None):
    """Download FRED series via public CSV — no API key required."""
    frames = []
    start_ts = pd.Timestamp(start)
    end_ts = pd.Timestamp(end) if end else pd.Timestamp.today()

    for name, series_id in series_map.items():
        url = f"https://fred.stlouisfed.org/graph/fredgraph.csv?id={series_id}"
        df = pd.read_csv(url, parse_dates=["observation_date"]).set_index("observation_date")
        col = [c for c in df.columns][0]
        s = pd.to_numeric(df[col], errors="coerce").rename(name)
        frames.append(s.loc[start_ts:end_ts])

    return pd.concat(frames, axis=1).sort_index()


def fetch_fred_panel(series_map, start, end=None, fred_client=None):
    if fred_client is not None:
        frames = []
        for name, series_id in series_map.items():
            s = fred_client.get_series(series_id, observation_start=start, observation_end=end)
            frames.append(s.rename(name))
        return pd.concat(frames, axis=1).sort_index()
    return fetch_fred_public_csv(series_map, start, end)


def to_monthly_macro(df):
    """Resample daily/weekly FRED series to month-end; forward-fill gaps."""
    monthly = df.resample("ME").last().ffill()
    monthly["yield_slope"] = monthly["dgs10"] - monthly["dgs2"]
    monthly["cpi_yoy"] = monthly["cpi"].pct_change(12) * 100
    monthly["unrate_chg"] = monthly["unrate"].diff()
    monthly["vix_chg"] = monthly["vix"].pct_change()
    return monthly


def fetch_asset_returns(tickers, start, end=None):
    data = yf.download(
        list(tickers.keys()),
        start=start,
        end=end,
        auto_adjust=True,
        progress=False,
        group_by="column",
    )
    # yfinance may return MultiIndex columns depending on version
    if isinstance(data.columns, pd.MultiIndex):
        prices = data["Close"].copy()
    else:
        prices = data["Close"].copy() if "Close" in data.columns else data.copy()

    if isinstance(prices, pd.Series):
        prices = prices.to_frame(name=list(tickers.keys())[0])

    prices = prices.rename(columns=str.upper)
    prices = prices[[t for t in tickers.keys() if t in prices.columns]]

    monthly_prices = prices.resample("ME").last()
    returns = monthly_prices.pct_change().dropna(how="all")
    return returns


macro_raw = fetch_fred_panel(FRED_SERIES, START_DATE, END_DATE, fred_client=fred)
macro = to_monthly_macro(macro_raw)
returns = fetch_asset_returns(ASSETS, START_DATE, END_DATE)

panel = macro.join(returns, how="inner").dropna()

# --- FIX 1: Clean bad return data (prevents backtest spikes) ---
asset_cols = list(ASSETS.keys())
MAX_MONTHLY_RETURN = 0.30  # cap any single asset month at +/-30%

bad_before = (panel[asset_cols].abs() > MAX_MONTHLY_RETURN).any(axis=1).sum()
panel[asset_cols] = panel[asset_cols].clip(-MAX_MONTHLY_RETURN, MAX_MONTHLY_RETURN)

# Drop the most recent month (often incomplete / noisy from Yahoo)
panel = panel.iloc[:-1]

print(f"Panel: {panel.index.min().date()} → {panel.index.max().date()} | {len(panel)} months")
print(f"Capped {bad_before} month(s) with returns beyond +/-{MAX_MONTHLY_RETURN:.0%}")
panel.tail()

## 2. Feature matrix for HMM

In [ ]:
FEATURE_COLS = [
    "yield_slope",  # curve shape
    "cpi_yoy",      # inflation
    "unrate",       # labor market
    "vix",          # risk appetite / stress
    "fedfunds",     # policy stance
]

features = panel[FEATURE_COLS].copy()
scaler = StandardScaler()
X = scaler.fit_transform(features)

train_mask = panel.index <= TRAIN_END
X_train = scaler.fit_transform(features.loc[train_mask])
X_all = scaler.transform(features)

features.describe().T

## 3. Fit Hidden Markov Model (3 states)

In [ ]:
def fit_hmm_3state(X_train, X_all, random_state=42, max_retries=25):
    """Fit 3-state HMM; retry with new seeds if training collapses to <3 states."""
    best_model, best_states, best_n = None, None, 0

    for attempt in range(max_retries):
        model = GaussianHMM(
            n_components=3,
            covariance_type="diag",  # more stable than 'full' on monthly macro data
            n_iter=3000,
            random_state=random_state + attempt,
            tol=1e-4,
            min_covar=1e-3,
        )
        model.fit(X_train)
        train_states = model.predict(X_train)
        n_unique = len(np.unique(train_states))

        if n_unique > best_n:
            best_n = n_unique
            best_model = model
            best_states = model.predict(X_all)

        if n_unique == 3:
            break

    print(
        f"Distinct training states: {best_n}/3 | "
        f"Converged: {best_model.monitor_.converged} | "
        f"Log-likelihood: {best_model.score(X_train):,.1f}"
    )
    if best_n < 3:
        print("Note: fewer than 3 states detected — labeling will use available states.")
    return best_model, best_states, best_n


hmm, raw_states, n_states = fit_hmm_3state(X_train, X_all, RANDOM_STATE)

In [ ]:
def label_regimes(features_df, states, feature_cols):
    """Map HMM state IDs to Expansion / Slowdown / Stress (robust to 1–3 states)."""
    summary = features_df.groupby(states)[feature_cols].mean()
    if summary.empty:
        raise ValueError("HMM produced no states. Re-run the HMM cell above.")

    stress_id = summary["vix"].idxmax()
    mapping = {stress_id: "Stress"}

    others = [i for i in summary.index if i != stress_id]
    if len(others) >= 1:
        expansion_id = summary.loc[others, "unrate"].idxmin()
        mapping[expansion_id] = "Expansion"
        others = [i for i in others if i != expansion_id]
    if len(others) >= 1:
        mapping[others[0]] = "Slowdown"
        for extra in others[1:]:
            mapping[extra] = "Slowdown"

    return summary, mapping


def smooth_states(states, window=3):
    """FIX 2: 3-month rolling mode — stops Slowdown/Stress flip-flopping every month."""
    s = pd.Series(states, dtype=int)

    def mode_or_current(x):
        vals = pd.Series(x).dropna().astype(int)
        if len(vals) == 0:
            return np.nan
        return vals.mode().iloc[0]

    return s.rolling(window, center=True, min_periods=1).apply(mode_or_current, raw=False).astype(int).values


raw_states_smooth = smooth_states(raw_states, window=3)
state_summary, state_map = label_regimes(features, raw_states_smooth, FEATURE_COLS)
regime = pd.Series(raw_states_smooth, index=panel.index).map(state_map)
regime.name = "regime"

print("State → Regime mapping:", state_map)
print("Regime counts:\n", regime.value_counts())
display(state_summary)

### FIX 3: Sanity check — did known crises label as Stress?

Run this after labeling. **Pass = Stress shows up in 2008 Q4 and/or 2020 Q1–Q2.**

In [ ]:
check_dates = {
    "GFC peak (2008 Q4)": ("2008-10-01", "2008-12-31"),
    "COVID crash (2020 Q1)": ("2020-02-01", "2020-04-30"),
    "COVID recovery (2020 Q2-Q3)": ("2020-05-01", "2020-09-30"),
}

print("Regime during known crisis windows:\n")
for label, (start, end) in check_dates.items():
    window = regime.loc[start:end]
    if window.empty:
        print(f"  {label}: no data")
        continue
    top = window.value_counts().idxmax()
    pct = window.value_counts(normalize=True).max() * 100
    flag = "PASS" if top == "Stress" and label.startswith(("GFC", "COVID crash")) else "CHECK"
    print(f"  [{flag}] {label}: mostly {top} ({pct:.0f}% of months)")
    print(f"         {window.value_counts().to_dict()}")

## 4. Regime timeline

In [ ]:
regime_colors = {"Expansion": "#2ecc71", "Slowdown": "#f39c12", "Stress": "#e74c3c"}

fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

axes[0].plot(panel.index, panel["dgs10"], label="10Y yield", color="steelblue")
axes[0].plot(panel.index, panel["dgs2"], label="2Y yield", color="lightsteelblue")
axes[0].set_ylabel("Yield (%)")
axes[0].legend(loc="upper right")
axes[0].set_title("Treasury Yields & Detected Macro Regime")

axes[1].plot(panel.index, panel["yield_slope"], color="purple", label="10Y − 2Y slope")
axes[1].axhline(0, color="black", lw=0.8, ls="--")
axes[1].set_ylabel("Slope (bp)")
axes[1].legend(loc="upper right")

axes[2].plot(panel.index, panel["vix"], color="crimson", label="VIX")
axes[2].set_ylabel("VIX")
axes[2].legend(loc="upper right")

for ax in axes:
    for i in range(len(regime) - 1):
        r = regime.iloc[i]
        if pd.isna(r) or r not in regime_colors:
            continue
        ax.axvspan(regime.index[i], regime.index[i + 1], color=regime_colors[r], alpha=0.12)

plt.tight_layout()
plt.show()

## 5. Cross-asset performance by regime

In [ ]:
asset_cols = list(ASSETS.keys())
regime_returns = panel[asset_cols].join(regime).groupby("regime").agg(["mean", "std"])
regime_returns.columns = [f"{a}_{s}" for a, s in regime_returns.columns]

ann_factor = 12
mean_tbl = panel[asset_cols].join(regime).groupby("regime").mean() * ann_factor
vol_tbl = panel[asset_cols].join(regime).groupby("regime").std() * np.sqrt(ann_factor)
sharpe_tbl = mean_tbl / vol_tbl

print("Annualized mean returns by regime")
display(mean_tbl.round(4))
print("\nAnnualized Sharpe (rf ≈ 0) by regime")
display(sharpe_tbl.round(4))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
mean_tbl.plot(kind="bar", ax=ax, rot=0)
ax.axhline(0, color="black", lw=0.8)
ax.set_title("Annualized Mean Returns by Macro Regime")
ax.set_ylabel("Return")
ax.legend(title="Asset", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

In [ ]:
transition = pd.crosstab(regime.shift(1), regime, normalize="index")
transition.index.name = "From"
transition.columns.name = "To"

plt.figure(figsize=(6, 4))
sns.heatmap(transition, annot=True, fmt=".2f", cmap="Blues", vmin=0, vmax=1)
plt.title("Regime Transition Probabilities (empirical)")
plt.tight_layout()
plt.show()

## 6. Out-of-sample backtest

**Rules (no look-ahead):**
- HMM trained only through `TRAIN_END`
- Regime signal lagged 1 month
- Asset ranking per regime uses **training-period** mean returns only
- Hold equal weight in top 2 assets for the active regime

In [ ]:
def regime_conditional_weights(regime_series, asset_returns, train_end, top_n=2):
    """Build monthly weights from lagged regime and train-set return rankings."""
    train_regime = regime_series.loc[:train_end]
    train_rets = asset_returns.loc[:train_end]

    rank_table = (
        train_rets.join(train_regime)
        .groupby("regime")
        .mean()
    )

    weights = pd.DataFrame(0.0, index=asset_returns.index, columns=asset_returns.columns)
    lagged_regime = regime_series.shift(1)

    for dt in asset_returns.index:
        if dt <= pd.Timestamp(train_end):
            continue
        r = lagged_regime.loc[dt]
        if pd.isna(r) or r not in rank_table.index:
            continue
        top_assets = rank_table.loc[r].nlargest(top_n).index.tolist()
        weights.loc[dt, top_assets] = 1.0 / top_n

    return weights, rank_table


weights, rank_table = regime_conditional_weights(regime, panel[asset_cols], TRAIN_END)
print("Training-period average monthly returns by regime (used for ranking):")
display(rank_table.round(4))

strategy_ret = (weights * panel[asset_cols]).sum(axis=1)
strategy_ret = strategy_ret.loc[TRAIN_END:].iloc[1:]  # OOS only

# FIX 4: Flag any remaining outlier months in the strategy
outliers = strategy_ret[strategy_ret.abs() > 0.15]
if len(outliers):
    print("WARNING — outlier strategy months (check data):")
    print(outliers.round(4))
    strategy_ret = strategy_ret.clip(-0.15, 0.15)
else:
    print("No outlier strategy months detected.")

bench_6040 = 0.6 * panel["SPY"] + 0.4 * panel["TLT"]
bench_6040 = bench_6040.loc[strategy_ret.index.min():]

ew_benchmark = panel[asset_cols].mean(axis=1).loc[strategy_ret.index.min():]

In [ ]:
def performance_stats(r, name):
    r = r.dropna()
    ann_ret = r.mean() * 12
    ann_vol = r.std() * np.sqrt(12)
    sharpe = ann_ret / ann_vol if ann_vol > 0 else np.nan
    cum = (1 + r).cumprod()
    max_dd = (cum / cum.cummax() - 1).min()
    return pd.Series(
        {"Strategy": name, "Ann Return": ann_ret, "Ann Vol": ann_vol, "Sharpe": sharpe, "Max DD": max_dd}
    )


stats = pd.DataFrame([
    performance_stats(strategy_ret, "Regime Strategy"),
    performance_stats(bench_6040, "60/40 SPY-TLT"),
    performance_stats(ew_benchmark, "Equal Weight 4-asset"),
]).set_index("Strategy")

print(f"Out-of-sample period: {strategy_ret.index.min().date()} → {strategy_ret.index.max().date()}")
display(stats.round(4))

In [ ]:
cum_strategy = (1 + strategy_ret).cumprod()
cum_6040 = (1 + bench_6040.loc[strategy_ret.index]).cumprod()
cum_ew = (1 + ew_benchmark.loc[strategy_ret.index]).cumprod()

plt.figure(figsize=(12, 5))
plt.plot(cum_strategy, label="Regime Strategy (OOS)", lw=2)
plt.plot(cum_6040, label="60/40 SPY-TLT", ls="--")
plt.plot(cum_ew, label="Equal Weight", ls=":")
plt.axvline(pd.Timestamp(TRAIN_END), color="gray", ls="--", alpha=0.7, label="Train end")
plt.title("Cumulative Return — Out-of-Sample Backtest")
plt.ylabel("Growth of $1")
plt.legend()
plt.tight_layout()
plt.show()

## 7. Key takeaways & resume material

Fill in your actual numbers after running the notebook.

In [ ]:
regime_pct = regime.value_counts(normalize=True).mul(100).round(1)
print("Time spent in each regime (% of sample):")
print(regime_pct.to_string())

print("\n--- Resume bullet (edit with your OOS stats) ---")
oos_sharpe = stats.loc["Regime Strategy", "Sharpe"]
bench_sharpe = stats.loc["60/40 SPY-TLT", "Sharpe"]
print(
    f"Built a 3-state macro regime engine (HMM) using FRED indicators and cross-asset ETFs; "
    f"backtested regime-conditional allocation OOS (Sharpe {oos_sharpe:.2f} vs 60/40 {bench_sharpe:.2f})."
)

## 8. Research memo outline (export to PDF for GitHub)

1. **Thesis:** Macro regimes cluster into expansion, slowdown, and stress with distinct cross-asset payoffs.
2. **Data:** FRED (yields, CPI, unemployment, Fed Funds, VIX) + SPY/TLT/GLD/UUP monthly returns.
3. **Method:** Gaussian HMM on standardized macro features; train through 2015, test OOS with 1-month lag.
4. **Findings:** [Which assets win in each regime? Any inversion → stress link?]
5. **Strategy:** Equal-weight top-2 historical performers per regime; report Sharpe, vol, max DD.
6. **Limitations:** Small sample, state labeling ambiguity, no transaction costs, parameter instability.
7. **PM extension:** Add Fed surprise index, credit spreads (BAA-AAA), vol-targeting, and sleeve-level risk limits.

---
*Educational project only — not investment advice.*